## Transform Constructors Data
1. Read bronze constructors table
2. Keep only the columns required for analytics(Drop url Column)
3. Standardise column names using snake_case (constructorId -> constructor_id)
4. Rename columns to make them more meaningful (name -> constructor_name)
5. Remove duplicate records
6. Transform values of columns nationality to Title Case
7. Write the transformed data to silver constructors table

In [0]:
%run ../00-common/01.environment-config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.constructors"
silver_table = f"{catalog_name}.{silver_schema}.constructors"

#### 1. Read bronze circuits table

In [0]:
constructors_df = spark.read.table(bronze_table)

In [0]:
display(constructors_df)

In [0]:
spark.read.option('versionAsOf',0).table(bronze_table)

#### 2. Keep only the columns required for analytics(Drop url Column)

In [0]:
from pyspark.sql import functions as F
constructors_selected_df = constructors_df.drop(F.col("url"))

#### Step 3 & 4 - Standardise column names
 - Standardise column names using snake_case (constructorId -> constructor_id)
 - Rename columns to make them more meaningful (name -> constructor_name)


In [0]:
constructors_renamed_df = (
    constructors_selected_df
    .withColumnsRenamed({
        "constructorId":"constructor_id",
        "name": "constructor_name"
    })
)

#### Step 5 - Remove duplicate records

In [0]:
constructors_distinct_df = constructors_renamed_df.dropDuplicates(["constructor_id"])

In [0]:
display(constructors_distinct_df)

#### Step 6 - Transform values of columns nationality to Title Case

In [0]:
constructors_final_df = (
    constructors_distinct_df
    .withColumn("nationality",F.initcap(F.col("nationality")))
)

In [0]:
display(constructors_final_df)

#### Step 7 -  Write the transformed data to silver circuits table

In [0]:
(
    constructors_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))